### MACS 30113/30123 Lab Session: Working with EMR Clusters/Spark
*Week 6*

**Adam Wu, Wonje Yun**


### Create S3 Bucket

First create S3 bucket to store our files. If you already have an S3 bucket that you can use for these purposes, you can skip the bucket setup that follows.

In [6]:
import boto3

In [ ]:
# Initialize boto3 handler
s3 = boto3.resource('s3')

# Create a new bucket to store your files
BUCKETNAME = 'adam-example-bucket'
s3.create_bucket(Bucket=BUCKETNAME)

# This is what we will use to interface with the specific bucket
bucket = s3.Bucket(BUCKETNAME)

In [ ]:
# Upload your .py file to S3
FILENAME = 'mystuff/myfile.py'
with open('mystuff/myfile.py', 'rb') as myfile:
    bucket.put_object(Key=FILENAME, Body=myfile)

### Launching EMR Cluster

Launch the EMR cluster from a terminal using `launch_spark_cluster.py` from the Week 7 Spark Intro materials (`in-class-activities/07_Spark_Intro/pyspark_basics/`). The script matches `emr_cheatsheet.md`: it creates a Spark-enabled cluster with Jupyter S3 persistence, opens SSH on the master node, then waits and prints `ssh` and port-forward commands when the cluster is ready.

In [ ]:
%%bash
# Must match BUCKETNAME in the "Create S3 Bucket" cell above.
S3_BUCKET="adam-example-bucket"

# Optional: pin region if your AWS CLI/default config is not us-east-1
export AWS_DEFAULT_REGION="${AWS_DEFAULT_REGION:-us-east-1}"

# Make sure you are in the right directory; or edit the file path to the python launch script
python launch_spark_cluster.py \
    --s3_bucket "$S3_BUCKET" \
    --primary_count 1 \
    --core_count 2 \
    --instance_type "m5.xlarge"


`launch_spark_cluster.py` adds an SSH rule on the master node security group when the cluster starts. If you still cannot connect, check the EMR primary node security group and your key pair (`vockey`) in the AWS console—see `emr_cheatsheet.md` in the Spark Intro materials.

#### Method 1: `ssh` Directly

The first way to work with EMR is to `ssh` into the primary node and work like a normal EC2 host (see the EC2 lab). Use the **master public DNS** from the EMR console (primary node) or copy the command printed when `launch_spark_cluster.py` finishes (it includes `hadoop@<MasterPublicDnsName>`).

Connecting:

```bash
ssh -i ~/.ssh/vockey.pem hadoop@MASTER_PUBLIC_DNS
```

Upload a local folder `mystuff` to the cluster:

```bash
scp -i ~/.ssh/vockey.pem -r mystuff hadoop@MASTER_PUBLIC_DNS:/home/hadoop
```

Download `mystuff` from the cluster to your current directory:

```bash
scp -i ~/.ssh/vockey.pem -r hadoop@MASTER_PUBLIC_DNS:/home/hadoop/mystuff .
```

---

After uploading your files in there, you can then run Spark jobs with
```bash
[EMR] spark-submit mystuff/myfile.py
```
Alternatively if your files are saved on `S3`, then
```bash
[EMR] spark-submit s3://adam-example-bucket/mystuff/myfile.py adam-example-bucket
```

#### Method 2: Interactive Sessions

You can use JupyterHub on the cluster. Forward port **9443** to your laptop (keep this terminal open), then open **`https://localhost:9443`** in a browser. Log in with username **`jovyan`** and password **`jupyter`** (defaults). Replace **`MASTER_PUBLIC_DNS`** with the primary node DNS from the EMR console or with the host shown in the port-forward command printed by `launch_spark_cluster.py`.

```bash
ssh -i ~/.ssh/vockey.pem -NL 9443:localhost:9443 hadoop@MASTER_PUBLIC_DNS
```


#### Method 3: Running Spark on Midway

Using `sbatch`, refer to `in-class-activities/07_Spark/7M_Spark_EDA_ML/midway` on Week 7 course materials. 

You can also work with Spark interactive on Midway with `sinteractive`:
```bash
$ sinteractive --ntasks=<NUMBER-OF-TASKS> --account=macs30123
```
This would take you into the computation node, where you can start `python3` or `ipython3` directly in the terminal. For Spark, you need to run `pyspark`.
```bash
$ pyspark
```
This would start the spark shell and pyspark, where you can work like `python3` in the terminal.
To exit from spark shell run:
```bash
$ exit()
```
To exit from sinteractive run:
```bash
$ exit()
```
